<a href="https://colab.research.google.com/github/nocleo/ADVLSI2_Project_updated/blob/main/notebooks/ADVLSI2_CNN_UNet_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Loading of Training dataset and data augmentation


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import os
import zipfile

# Unzip dataset
with zipfile.ZipFile('training_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

class DRCDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = ['clean', 'dirty']
        self.file_list = []
        for idx, cls in enumerate(self.classes):
            path = os.path.join(self.root_dir, cls)
            for f in os.listdir(path):
                if f.endswith('.npy'):
                    self.file_list.append((os.path.join(path, f), idx))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path, label = self.file_list[idx]
        matrix = np.load(file_path).astype(np.float32)
        matrix = torch.from_numpy(matrix).unsqueeze(0) # Channel dimension
        if self.transform:
            matrix = self.transform(matrix)
        return matrix, label

# Industry Standard Augmentation + Random Shift
data_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    # Translation of up to 10% (20px) to prevent spatial bias
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))
])

In [ ]:
# CNN Arcitechture defintion according to the article


class NCSU_DRCNN(nn.Module):
    def __init__(self):
        super(NCSU_DRCNN, self).__init__()
        # Conv1: 32 filters, 3x3
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # 200 -> 100
        )
        # Conv2: 16 filters, 3x3
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # 100 -> 50
        )
        # Conv3: 16 filters, 3x3
        self.conv3 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # 50 -> 25
        )
        # Conv4: 32 filters, 3x3
        self.conv4 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # 25 -> 12
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, 2) # Binary Classification
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
# Model Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NCSU_DRCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

# Loading and Splitting
dataset = DRCDataset(root_dir='data', transform=data_transforms)
train_size = int(0.80 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Total samples: {len(dataset)}")
print(f"Training: {len(train_data)} | Validation: {len(val_data)} | Testing: {len(test_data)}")


def train(epochs=20):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Simple Accuracy Check
        model.eval()
        correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                _, pred = torch.max(model(imgs), 1)
                correct += (pred == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {100*correct/len(val_data):.2f}%")

train(epochs=40)

# Saving model
torch.save(model.state_dict(), 'ncsu_drcnn_weights.pth')
print("Training complete! Model saved to 'ncsu_drcnn_weights.pth'")

In [ ]:
# Geenration of confusion matrix

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Switch to evaluation mode
model.eval()
all_preds = []
all_labels = []

# Final inference on the Testing Set (the 5%)
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate Final Accuracy
cm = confusion_matrix(all_labels, all_preds)
accuracy = (cm[0,0] + cm[1,1]) / sum(sum(cm))

# Visualization of the Confusion Matrix (similar to Figure 7 in paper)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean (NDRC)', 'Violation (DRCV)'],
            yticklabels=['Clean (NDRC)', 'Violation (DRCV)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Final Testing Confusion Matrix\nAccuracy: {accuracy*100:.2f}%')
plt.show()

# Print full report (Recall and Precision)
print("\nFinal Performance Report:")
print(classification_report(all_labels, all_preds, target_names=['Clean', 'Violation']))

In [ ]:

# Inference - model excecution on inference dataset

import os
import numpy as np
import torch
import time
import zipfile
import io

# --- CONFIGURATION ---
# Point directly to the ZIP file shown in your Colab root directory
ZIP_PATH ="/content/inference_dataset.zip"
CONFIDENCE_THRESHOLD = 0.80

def scan_layout_from_zip(model, zip_file_path, threshold=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    if not os.path.exists(zip_file_path):
        print(f"[!] Error: Could not find ZIP file at {zip_file_path}")
        return

    print(f"[*] Opening ZIP archive at {zip_file_path}...")

    violations = []
    start_time = time.time()

    # Open the ZIP file directly
    with zipfile.ZipFile(zip_file_path, 'r') as archive:
        # Filter only .npy files from the archive list
        npy_files = [f for f in archive.namelist() if f.endswith('.npy')]

        if not npy_files:
            print("[!] Error: No .npy files found inside the ZIP.")
            return

        print(f"[*] Starting DRC scan on {len(npy_files)} tiles directly from ZIP using {device}...")

        with torch.no_grad():
            for i, filename in enumerate(npy_files):
                # Read the raw bytes from the ZIP into memory
                with archive.open(filename) as f:
                    file_bytes = f.read()
                    # Convert bytes directly to a NumPy array
                    matrix = np.load(io.BytesIO(file_bytes)).astype(np.float32)

                # Format for CNN: (1, 1, 200, 200)
                tensor = torch.from_numpy(matrix).unsqueeze(0).unsqueeze(0).to(device)

                outputs = model(tensor)
                probs = torch.softmax(outputs, dim=1)
                violation_prob = probs[0][1].item()

                if violation_prob >= threshold:
                    # Extract pure filename without folder paths that might be in the ZIP
                    base_name = os.path.basename(filename)
                    coords = base_name.replace("tile_", "").replace(".npy", "")
                    violations.append((coords, violation_prob))

                if (i + 1) % 2000 == 0:
                    print(f"[>] Scanned {i + 1}/{len(npy_files)} tiles...")

    end_time = time.time()
    print(f"\n[+] Scan complete in {end_time - start_time:.1f} seconds!")

    # --- Print results and save report ---
    print("=" * 40)
    if not violations:
        print("[*] LAYOUT IS CLEAN! No DRC violations found.")
    else:
        print(f"[!] FOUND {len(violations)} POTENTIAL VIOLATIONS!")
        print("=" * 40)

        with open("drc_report.txt", "w") as f:
            f.write("CNN DRC Violations Report\n")
            f.write("=========================\n")
            for coords, prob in violations:
                line = f"Location: {coords} | Confidence: {prob:.2%}"
                print(f"  - {line}")
                f.write(line + "\n")

        print("\n[-] Detailed report saved to 'drc_report.txt' in Colab files.")

# --- 1. Model Initialization ---
model = NCSU_DRCNN()

# --- 2. Load Trained Weights ---
model_weights_path = 'ncsu_drcnn_weights.pth'

model.load_state_dict(torch.load(model_weights_path, map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu'), weights_only=True))

# --- 3. Execute Inference ---
scan_layout_from_zip(model, ZIP_PATH, CONFIDENCE_THRESHOLD)


**checking Semantic Segmentation option**

1. checking how the data that we have looks

In [ ]:
import os

print(os.listdir("."))

In [ ]:
import zipfile

zip_path = "training_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    files = z.namelist()

print("Number of files:", len(files))
print("First 30 files:")
for f in files[:30]:
    print(f)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import io

with zipfile.ZipFile("training_dataset.zip", 'r') as z:
    dirty_files = [f for f in z.namelist() if f.startswith("dirty/") and f.endswith(".npy")]
    clean_files = [f for f in z.namelist() if f.startswith("clean/") and f.endswith(".npy")]

    print("Dirty samples:", len(dirty_files))
    print("Clean samples:", len(clean_files))
    print("Example dirty file:", dirty_files[0])
    print("Example clean file:", clean_files[0])

    with z.open(dirty_files[0]) as f:
        dirty_arr = np.load(io.BytesIO(f.read()))

    with z.open(clean_files[0]) as f:
        clean_arr = np.load(io.BytesIO(f.read()))

print("Dirty shape:", dirty_arr.shape)
print("Dirty unique values:", np.unique(dirty_arr))
print("Clean shape:", clean_arr.shape)
print("Clean unique values:", np.unique(clean_arr))

plt.figure(figsize=(5,5))
plt.imshow(dirty_arr, cmap="gray")
plt.title("Example dirty tile")
plt.axis("off")
plt.show()

plt.figure(figsize=(5,5))
plt.imshow(clean_arr, cmap="gray")
plt.title("Example clean tile")
plt.axis("off")
plt.show()

2. i edit the generat_training_datasat.py so that it will also generate images and masks. reupload the zip folder it created. now run this code to see how the data generated by the code looks now

In [ ]:
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import io

zip_path = "training_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    image_files = sorted([f for f in z.namelist() if f.startswith("images/") and f.endswith(".npy")])
    mask_files = sorted([f for f in z.namelist() if f.startswith("masks/") and f.endswith(".npy")])

    print("Images:", len(image_files))
    print("Masks:", len(mask_files))
    print("Example image:", image_files[0])
    print("Example mask:", mask_files[0])

    # find an example with a non-empty mask
    example_idx = None
    for i, mf in enumerate(mask_files):
        with z.open(mf) as f:
            mask = np.load(io.BytesIO(f.read()))
        if mask.sum() > 0:
            example_idx = i
            break

    print("First non-empty mask index:", example_idx)
    print("Mask file:", mask_files[example_idx])
    print("Image file:", image_files[example_idx])

    with z.open(image_files[example_idx]) as f:
        img = np.load(io.BytesIO(f.read()))

    with z.open(mask_files[example_idx]) as f:
        mask = np.load(io.BytesIO(f.read()))

print("Image shape:", img.shape, "unique:", np.unique(img))
print("Mask shape:", mask.shape, "unique:", np.unique(mask), "mask sum:", mask.sum())

plt.figure(figsize=(5,5))
plt.imshow(img, cmap="gray")
plt.title("Layout image")
plt.axis("off")
plt.show()

plt.figure(figsize=(5,5))
plt.imshow(mask, cmap="gray")
plt.title("DRC violation mask")
plt.axis("off")
plt.show()

plt.figure(figsize=(5,5))
plt.imshow(img, cmap="gray")
plt.imshow(mask, cmap="Reds", alpha=0.6)
plt.title("Overlay: image + mask")
plt.axis("off")
plt.show()

3. ### Sanity Check 1: Verify that clean tiles have empty masks

In this step, we verify that tiles labeled as clean do not contain any DRC violation pixels in their corresponding masks.  
For a correct segmentation dataset, every clean tile should have a mask with only zeros.

In [ ]:
import zipfile
import numpy as np
import io

zip_path = "training_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    clean_files = sorted([f for f in z.namelist() if f.startswith("clean/") and f.endswith(".npy")])
    mask_files = sorted([f for f in z.namelist() if f.startswith("masks/") and f.endswith(".npy")])

    clean_nonzero_masks = 0
    checked = 0

    for cf in clean_files[:100]:
        fname = cf.replace("clean/", "")
        mf = "masks/" + fname

        if mf in mask_files:
            with z.open(mf) as f:
                mask = np.load(io.BytesIO(f.read()))
            checked += 1

            if mask.sum() > 0:
                clean_nonzero_masks += 1

print("Checked clean masks:", checked)
print("Clean masks with non-zero pixels:", clean_nonzero_masks)

4. ### Sanity Check 2: Count empty and non-empty segmentation masks

In this step, we check the distribution of the generated masks.  
A non-empty mask corresponds to a tile that contains a DRC violation, while an empty mask corresponds to a clean tile.  
This verifies that the number of non-empty masks is consistent with the number of dirty samples.

In [ ]:
import zipfile
import numpy as np
import io

zip_path = "training_dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    mask_files = sorted([f for f in z.namelist() if f.startswith("masks/") and f.endswith(".npy")])

    non_empty = 0
    empty = 0
    sums = []

    for mf in mask_files:
        with z.open(mf) as f:
            mask = np.load(io.BytesIO(f.read()))

        s = mask.sum()
        sums.append(s)

        if s > 0:
            non_empty += 1
        else:
            empty += 1

print("Total masks:", len(mask_files))
print("Non-empty masks:", non_empty)
print("Empty masks:", empty)
print("Min mask sum:", min(sums))
print("Max mask sum:", max(sums))
print("Mean mask sum:", sum(sums) / len(sums))

5. the dataset is ready for Semantic Segmentation

### Segmentation Dataset Class

In this step, we define a PyTorch Dataset for the semantic segmentation task.  
Each sample consists of a layout image tile and its corresponding DRC violation mask.  
Unlike the previous classification dataset, where each tile had a single clean/dirty label, here each tile has a pixel-level target mask.

In [ ]:
import zipfile
import numpy as np
import io
import torch
from torch.utils.data import Dataset

class DRCSegmentationDataset(Dataset):
    def __init__(self, zip_path):
        self.zip_path = zip_path

        with zipfile.ZipFile(self.zip_path, 'r') as z:
            self.image_files = sorted([
                f for f in z.namelist()
                if f.startswith("images/") and f.endswith(".npy")
            ])

            self.mask_files = sorted([
                f for f in z.namelist()
                if f.startswith("masks/") and f.endswith(".npy")
            ])

        assert len(self.image_files) == len(self.mask_files), "Number of images and masks must match"

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        mask_file = self.mask_files[idx]

        with zipfile.ZipFile(self.zip_path, 'r') as z:
            with z.open(image_file) as f:
                image = np.load(io.BytesIO(f.read()))

            with z.open(mask_file) as f:
                mask = np.load(io.BytesIO(f.read()))

        # Convert from numpy arrays to PyTorch tensors
        # Shape: [H, W] -> [1, H, W]
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

### Dataset Sanity Check

Here we create an instance of the segmentation dataset and load one sample.  
We verify that both the input image and the target mask have the expected shape: `[1, 200, 200]`, where the first dimension is the channel dimension.

In [ ]:
zip_path = "training_dataset.zip"

seg_dataset = DRCSegmentationDataset(zip_path)

print("Dataset size:", len(seg_dataset))

image, mask = seg_dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image min/max:", image.min().item(), image.max().item())
print("Mask min/max:", mask.min().item(), mask.max().item())
print("Mask sum:", mask.sum().item())

### Train / Validation / Test Split

In this step, we split the segmentation dataset into training, validation, and test sets.  
The training set is used to optimize the U-Net model, the validation set is used to monitor performance during training, and the test set will be used only at the end for final evaluation.

In [ ]:
from torch.utils.data import random_split, DataLoader

# Dataset split ratios
train_ratio = 0.80
val_ratio = 0.15
test_ratio = 0.05

total_size = len(seg_dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size

# Reproducible split
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    seg_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print("Total samples:", total_size)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

# DataLoaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Visualize a Batch from the Segmentation DataLoader

Before training the model, we visualize a few examples from the training DataLoader.  
This helps verify that the input images and target masks are correctly paired and that the masks align with the DRC violation regions.

In [ ]:
import matplotlib.pyplot as plt

# Get one batch from the training loader
images, masks = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch mask shape:", masks.shape)

num_examples = 4

for i in range(num_examples):
    img = images[i, 0].numpy()
    mask = masks[i, 0].numpy()

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img, cmap="gray")
    plt.title("Layout image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Ground-truth mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img, cmap="gray")
    plt.imshow(mask, cmap="Reds", alpha=0.6)
    plt.title("Overlay")
    plt.axis("off")

    plt.show()

### U-Net Model for DRC Violation Segmentation

In this step, we define a basic U-Net model for semantic segmentation.  
The model receives a binary layout image tile as input and predicts a pixel-level DRC violation mask.  
The encoder extracts spatial features from the layout, while the decoder upsamples the features back to the original image resolution. Skip connections help preserve fine geometric details that are important for DRC localization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """
    Two consecutive convolution layers with ReLU activations.
    This block is used throughout the U-Net encoder and decoder.
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    """
    Basic U-Net architecture for binary semantic segmentation.
    Input:  [batch, 1, 200, 200]
    Output: [batch, 1, 200, 200]
    """
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Decoder
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        # Final 1x1 convolution to produce a binary mask logit
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)          # [B, 32, 200, 200]
        e2 = self.enc2(self.pool(e1))  # [B, 64, 100, 100]
        e3 = self.enc3(self.pool(e2))  # [B, 128, 50, 50]

        # Bottleneck
        b = self.bottleneck(self.pool(e3))  # [B, 256, 25, 25]

        # Decoder with skip connections
        d3 = self.up3(b)              # [B, 128, 50, 50]
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)             # [B, 64, 100, 100]
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)             # [B, 32, 200, 200]
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        logits = self.final_conv(d1)
        return logits

### Initialize the U-Net Model

Here we initialize the U-Net model and move it to GPU if available.  
The model outputs raw logits, which will later be converted to probabilities using a sigmoid function.

In [ ]:
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(in_channels=1, out_channels=1).to(device)

print("Using device:", device)

if os.path.exists("unet_drc_segmentation.pth"):
    state_dict = torch.load("unet_drc_segmentation.pth", map_location=device)
    model.load_state_dict(state_dict)
    print("Loaded trained U-Net weights from unet_drc_segmentation.pth")
else:
    print("No saved model found. Model initialized with random weights.")

model.eval()

# Test one forward pass
images, masks = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

### Segmentation Loss Function

For semantic segmentation, the model predicts a probability for each pixel.  
We combine Binary Cross Entropy loss with Dice loss.  
Binary Cross Entropy helps the model learn pixel-wise classification, while Dice loss helps handle the strong class imbalance between background pixels and small DRC violation regions.

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice loss for binary segmentation.
    It measures the overlap between the predicted mask and the ground-truth mask.
    """
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # Convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Flatten tensors
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)

        return 1.0 - dice.mean()


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def segmentation_loss(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)

### Training and Validation Functions

In this step, we define helper functions for training and validation.  
During training, the model updates its weights using the segmentation loss.  
During validation, the model is evaluated without weight updates, allowing us to monitor whether it generalizes beyond the training data.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = segmentation_loss(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


def validate_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = segmentation_loss(outputs, masks)

            total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

### Train the U-Net Model

Here we train the U-Net model on the DRC segmentation dataset.  
The training loss measures how well the model predicts the violation masks on training samples, while the validation loss measures performance on unseen validation samples.

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate_one_epoch(model, val_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

torch.save(model.state_dict(), "unet_drc_segmentation.pth")
print("Model saved to unet_drc_segmentation.pth")

# Evaluate the U-Net Segmentation Performance

After training, we evaluate the U-Net model on the test dataset.
We compute quantitative segmentation metrics including Dice Score, Intersection over Union (IoU), and Centroid Error to measure the quality of DRC violation localization.

In [ ]:
import numpy as np
import torch

def compute_metrics(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()

    dice = (2 * intersection + eps) / (pred.sum() + target.sum() + eps)
    iou = (intersection + eps) / (union + eps)

    return dice, iou


def centroid(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    return np.array([xs.mean(), ys.mean()])


def centroid_error(pred, target):
    c_pred = centroid(pred)
    c_target = centroid(target)

    if c_pred is None or c_target is None:
        return None

    return np.linalg.norm(c_pred - c_target)


def evaluate_segmentation(model, loader, device, threshold=0.5):
    model.eval()

    dice_scores = []
    iou_scores = []
    centroid_errors = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()

            preds_np = preds.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for i in range(preds_np.shape[0]):
                pred_mask = preds_np[i, 0]
                true_mask = masks_np[i, 0]

                dice, iou = compute_metrics(pred_mask, true_mask)
                dice_scores.append(dice)
                iou_scores.append(iou)

                ce = centroid_error(pred_mask, true_mask)
                if ce is not None:
                    centroid_errors.append(ce)

    return {
        "threshold": threshold,
        "dice": np.mean(dice_scores),
        "iou": np.mean(iou_scores),
        "centroid_error": np.mean(centroid_errors) if centroid_errors else None
    }


for th in [0.3, 0.5, 0.7]:
    results = evaluate_segmentation(model, test_loader, device, threshold=th)
    print(results)

### Training Loss Curves

This plot shows the training and validation loss over epochs.  
A decreasing loss indicates that the U-Net is learning to predict the DRC violation masks.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("U-Net Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

### Visualize U-Net Predictions

In this step, we visualize the segmentation results produced by the trained U-Net model.  
For each example, we show the input layout image, the ground-truth DRC mask, the predicted probability map, and the predicted binary mask after thresholding.  
This helps us understand whether the model learned to localize the DRC violation regions.

In [ ]:
import matplotlib.pyplot as plt
import torch

model.eval()

# Get one batch from the validation set
images, masks = next(iter(val_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    logits = model(images)
    probs = torch.sigmoid(logits)

# Convert to CPU for visualization
images_cpu = images.cpu()
masks_cpu = masks.cpu()
probs_cpu = probs.cpu()

threshold = 0.5
pred_masks_cpu = (probs_cpu > threshold).float()

num_examples = 4

for i in range(num_examples):
    img = images_cpu[i, 0].numpy()
    gt_mask = masks_cpu[i, 0].numpy()
    prob_map = probs_cpu[i, 0].numpy()
    pred_mask = pred_masks_cpu[i, 0].numpy()

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(img, cmap="gray")
    plt.title("Layout image")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(gt_mask, cmap="gray")
    plt.title("Ground-truth mask")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(prob_map, cmap="hot")
    plt.title("Predicted probability")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(img, cmap="gray")
    plt.imshow(pred_mask, cmap="Reds", alpha=0.6)
    plt.title("Predicted mask overlay")
    plt.axis("off")

    plt.show()

In [ ]:
import os
print(os.listdir("."))

### Download Trained U-Net Model

This cell downloads the trained U-Net weights so the model can be loaded again in a future session without retraining from scratch.

In [ ]:
from google.colab import files

files.download("unet_drc_segmentation.pth")

### Current Progress

We successfully converted the original clean/dirty classification dataset into a semantic segmentation dataset with paired layout images and DRC masks.

Current dataset:

* Images: 2627
* Masks: 2627
* Clean samples contain empty masks
* Dirty samples contain DRC violation regions

We implemented and trained a basic U-Net model for 10 epochs and saved the trained model as `unet_drc_segmentation.pth`.

The model was evaluated using several segmentation metrics and prediction thresholds.

Evaluation results:

| Threshold | Dice Score | IoU   | Centroid Error |
| --------- | ---------- | ----- | -------------- |
| 0.3       | 0.859      | 0.851 | 4.23 px        |
| 0.5       | 0.890      | 0.882 | 4.08 px        |
| 0.7       | 0.898      | 0.891 | 3.89 px        |

The results indicate that the U-Net is able to accurately localize DRC violation regions with high overlap accuracy and low localization error, demonstrating that semantic segmentation is a promising approach for DRC hotspot localization.

### Next Steps

1. Compare the U-Net localization results with the CNN + GradCAM approach using both visual examples and quantitative metrics.

2. Compute additional detection metrics such as Precision, Recall, and F1-score.

3. Analyze representative false-positive and false-negative prediction examples to better understand model limitations.

4. Explore potential improvements to the current U-Net model, such as additional training epochs, data augmentation, or architectural enhancements.

5. Prepare figures, tables, and summary results for the final presentation and project report.
.

### Threshold Sweep and Pixel-to-Nanometer Conversion

In this step, we evaluate the trained U-Net model using multiple threshold values.  
The model outputs a probability map, and each threshold converts this probability map into a binary predicted mask.

For each threshold, we compute:
- Dice Score
- IoU
- Average Centroid Error in pixels
- Average Centroid Error in nanometers

Since each tile represents a physical window of 1600nm and is rasterized into a 200x200 image, the resolution is:

`1600nm / 200px = 8nm per pixel`

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Physical resolution settings
PHYSICAL_SIZE_NM = 1600
IMAGE_SIZE_PX = 200
NM_PER_PIXEL = PHYSICAL_SIZE_NM / IMAGE_SIZE_PX

print(f"Resolution: {NM_PER_PIXEL:.2f} nm/pixel")

### Evaluation Function for Dice, IoU, and Centroid Error

This function evaluates the segmentation model for a given threshold.  
Dice and IoU measure the overlap between the predicted mask and the ground-truth mask.  
Centroid Error measures the distance between the center of the predicted violation region and the center of the true violation region.

In [ ]:
def compute_centroid(mask):
    """
    Compute centroid of a binary mask.
    Returns None if the mask is empty.
    """
    coords = torch.nonzero(mask > 0, as_tuple=False)

    if coords.numel() == 0:
        return None

    # coords format: [N, 2] -> rows, cols = y, x
    centroid = coords.float().mean(dim=0)
    return centroid


def evaluate_segmentation_at_threshold(model, loader, device, threshold=0.5):
    model.eval()

    total_intersection = 0.0
    total_union = 0.0
    total_pred_sum = 0.0
    total_target_sum = 0.0

    centroid_errors_px = []

    gt_non_empty_count = 0
    pred_non_empty_count = 0
    missed_gt_count = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()

            # Global Dice / IoU over the full batch
            intersection = (preds * masks).sum()
            union = ((preds + masks) > 0).float().sum()

            total_intersection += intersection.item()
            total_union += union.item()
            total_pred_sum += preds.sum().item()
            total_target_sum += masks.sum().item()

            # Per-sample centroid error
            for i in range(images.size(0)):
                pred_mask = preds[i, 0]
                gt_mask = masks[i, 0]

                gt_centroid = compute_centroid(gt_mask)
                pred_centroid = compute_centroid(pred_mask)

                if gt_centroid is not None:
                    gt_non_empty_count += 1

                    if pred_centroid is not None:
                        pred_non_empty_count += 1
                        error_px = torch.norm(pred_centroid - gt_centroid).item()
                        centroid_errors_px.append(error_px)
                    else:
                        missed_gt_count += 1

    dice = (2 * total_intersection) / (total_pred_sum + total_target_sum + 1e-8)
    iou = total_intersection / (total_union + 1e-8)

    if len(centroid_errors_px) > 0:
        avg_centroid_error_px = np.mean(centroid_errors_px)
        avg_centroid_error_nm = avg_centroid_error_px * NM_PER_PIXEL
    else:
        avg_centroid_error_px = np.nan
        avg_centroid_error_nm = np.nan

    return {
        "threshold": threshold,
        "dice": dice,
        "iou": iou,
        "centroid_error_px": avg_centroid_error_px,
        "centroid_error_nm": avg_centroid_error_nm,
        "gt_non_empty": gt_non_empty_count,
        "pred_non_empty": pred_non_empty_count,
        "missed_gt": missed_gt_count
    }

### Prediction Probability Sanity Check

Before running the threshold sweep, we verify that the trained model was loaded correctly.  
A trained model should produce meaningful probability values, including values above the selected thresholds.

In [ ]:
images, masks = next(iter(val_loader))

images = images.to(device)
masks = masks.to(device)

model.eval()

with torch.no_grad():
    logits = model(images)
    probs = torch.sigmoid(logits)

print("Probability min:", probs.min().item())
print("Probability max:", probs.max().item())
print("Probability mean:", probs.mean().item())
print("GT mask sum:", masks.sum().item())

### Threshold Sweep on the Validation Set

Here we evaluate multiple threshold values on the validation set.  
This helps us choose the threshold that gives the best segmentation quality and localization accuracy.

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)

results = []

for th in thresholds:
    metrics = evaluate_segmentation_at_threshold(
        model=model,
        loader=val_loader,
        device=device,
        threshold=float(th)
    )
    results.append(metrics)

results_df = pd.DataFrame(results)

results_df

### Plot Metrics as a Function of Threshold

These plots show how the segmentation performance changes as a function of the threshold.  
Higher Dice and IoU indicate better overlap with the ground-truth masks.  
Lower Centroid Error indicates better localization accuracy.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["dice"], marker="o", label="Dice Score")
plt.plot(results_df["threshold"], results_df["iou"], marker="o", label="IoU")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Dice and IoU vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["centroid_error_px"], marker="o", label="Centroid Error [px]")
plt.xlabel("Threshold")
plt.ylabel("Centroid Error [pixels]")
plt.title("Centroid Error vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["centroid_error_nm"], marker="o", label="Centroid Error [nm]")
plt.xlabel("Threshold")
plt.ylabel("Centroid Error [nm]")
plt.title("Centroid Error vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

### Best Threshold Selection

In this step, we select the threshold with the highest Dice score.  
This threshold provides the best overlap between the predicted masks and the ground-truth DRC masks on the validation set.

In [ ]:
best_idx = results_df["dice"].idxmax()
best_row = results_df.loc[best_idx]

print("Best threshold based on Dice Score:")
print(f"Threshold: {best_row['threshold']:.2f}")
print(f"Dice: {best_row['dice']:.4f}")
print(f"IoU: {best_row['iou']:.4f}")
print(f"Centroid Error: {best_row['centroid_error_px']:.2f} px")
print(f"Centroid Error: {best_row['centroid_error_nm']:.2f} nm")
print(f"Missed GT masks: {int(best_row['missed_gt'])}")

### Final Evaluation on the Test Set

After selecting the best threshold using the validation set, we evaluate the final model on the held-out test set.  
The test set was not used for training or threshold selection, so it provides a more reliable estimate of the model's final segmentation and localization performance.

In [ ]:
best_threshold = float(best_row["threshold"])

test_metrics = evaluate_segmentation_at_threshold(
    model=model,
    loader=test_loader,
    device=device,
    threshold=best_threshold
)

print("Final Test Set Evaluation")
print(f"Threshold: {test_metrics['threshold']:.2f}")
print(f"Dice: {test_metrics['dice']:.4f}")
print(f"IoU: {test_metrics['iou']:.4f}")
print(f"Centroid Error: {test_metrics['centroid_error_px']:.2f} px")
print(f"Centroid Error: {test_metrics['centroid_error_nm']:.2f} nm")
print(f"GT non-empty masks: {test_metrics['gt_non_empty']}")
print(f"Predicted non-empty masks: {test_metrics['pred_non_empty']}")
print(f"Missed GT masks: {test_metrics['missed_gt']}")

### Final Results Interpretation

The final evaluation on the held-out test set shows strong segmentation and localization performance.

Using the validation-selected threshold of 0.9, the model achieved:
- Dice Score: 0.9469
- IoU: 0.8991
- Average Centroid Error: 3.77 pixels
- Average Centroid Error: 30.17 nm
- Missed ground-truth violations: 0

These results indicate that the U-Net model successfully predicts pixel-level DRC violation masks and localizes the violation regions with high accuracy.  
The average localization error of approximately 30 nm suggests that semantic segmentation can provide a meaningful improvement over tile-level classification, which only predicts whether a tile is clean or dirty.

It is important to note that this is still a proof-of-concept evaluation using a random tile-level split.  
A stronger future evaluation should test generalization across different layouts, for example by training on some layouts and testing on unseen layouts.

### Visualize Test Examples with DRC Violations

Here we visualize only test samples that contain non-empty ground-truth DRC masks.  
This makes it easier to inspect whether the model correctly localizes actual violations.

In [ ]:
import matplotlib.pyplot as plt
import torch

best_threshold = float(best_row["threshold"])
print(f"Using threshold: {best_threshold:.2f}")

model.eval()

shown = 0
num_examples = 6

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)
        pred_masks = (probs > best_threshold).float()

        images_cpu = images.cpu()
        masks_cpu = masks.cpu()
        probs_cpu = probs.cpu()
        pred_masks_cpu = pred_masks.cpu()

        for i in range(images_cpu.size(0)):
            gt_mask = masks_cpu[i, 0].numpy()

            # Show only examples with real DRC violations
            if gt_mask.sum() == 0:
                continue

            img = images_cpu[i, 0].numpy()
            prob_map = probs_cpu[i, 0].numpy()
            pred_mask = pred_masks_cpu[i, 0].numpy()

            plt.figure(figsize=(18, 4))

            plt.subplot(1, 4, 1)
            plt.imshow(img, cmap="gray")
            plt.title("Layout image")
            plt.axis("off")

            plt.subplot(1, 4, 2)
            plt.imshow(gt_mask, cmap="gray")
            plt.title("Ground-truth mask")
            plt.axis("off")

            plt.subplot(1, 4, 3)
            plt.imshow(prob_map, cmap="hot")
            plt.title("Predicted probability")
            plt.axis("off")

            plt.subplot(1, 4, 4)
            plt.imshow(img, cmap="gray")
            plt.imshow(pred_mask, cmap="Reds", alpha=0.6)
            plt.title("Predicted mask overlay")
            plt.axis("off")

            plt.show()

            shown += 1
            if shown >= num_examples:
                break

        if shown >= num_examples:
            break

### Extract Numerical Location from Predicted Masks

In this step, we convert the predicted binary DRC masks into numerical locations.

For each predicted mask, we extract:
- Centroid: the center point of the predicted violation region
- Bounding box: the smallest rectangle containing the predicted violation region
- Confidence: the average predicted probability inside the predicted mask

The tile filename contains the layout origin coordinates, for example:
`tile_100310_55800.npy`

Using the image resolution of 8nm/pixel, we convert pixel coordinates back into approximate layout coordinates in nanometers.

In [ ]:
import re
import numpy as np
import pandas as pd
import torch

# Resolution settings
PHYSICAL_SIZE_NM = 1600
IMAGE_SIZE_PX = 200
NM_PER_PIXEL = PHYSICAL_SIZE_NM / IMAGE_SIZE_PX

best_threshold = float(best_row["threshold"])

print(f"Using threshold: {best_threshold:.2f}")
print(f"Resolution: {NM_PER_PIXEL:.2f} nm/pixel")


def parse_tile_origin_from_filename(filename):
    """
    Extract tile origin coordinates from filenames like:
    images/tile_100310_55800.npy
    """
    match = re.search(r"tile_(\d+)_(\d+)\.npy", filename)
    if match is None:
        return None, None

    tile_x = int(match.group(1))
    tile_y = int(match.group(2))
    return tile_x, tile_y


def extract_mask_location(mask, prob_map=None):
    """
    Extract centroid and bounding box from a binary predicted mask.
    mask shape: [H, W]
    prob_map shape: [H, W], optional
    """
    ys, xs = np.where(mask > 0)

    if len(xs) == 0:
        return None

    # Centroid in pixel coordinates
    centroid_x_px = xs.mean()
    centroid_y_px = ys.mean()

    # Bounding box in pixel coordinates
    x_min_px = xs.min()
    x_max_px = xs.max()
    y_min_px = ys.min()
    y_max_px = ys.max()

    # Confidence: average predicted probability inside the predicted mask
    if prob_map is not None:
        confidence = prob_map[mask > 0].mean()
        max_probability = prob_map[mask > 0].max()
    else:
        confidence = np.nan
        max_probability = np.nan

    return {
        "centroid_x_px": centroid_x_px,
        "centroid_y_px": centroid_y_px,
        "x_min_px": x_min_px,
        "x_max_px": x_max_px,
        "y_min_px": y_min_px,
        "y_max_px": y_max_px,
        "confidence": confidence,
        "max_probability": max_probability
    }


def pixel_to_layout_coordinates(tile_x, tile_y, centroid_x_px, centroid_y_px):
    """
    Convert centroid pixel coordinates back to approximate layout coordinates.

    Note:
    The image matrix was flipped vertically using np.flipud during dataset generation.
    Therefore, image row 0 corresponds to the top of the physical tile.
    """
    layout_x_nm = tile_x + centroid_x_px * NM_PER_PIXEL
    layout_y_nm = tile_y + (IMAGE_SIZE_PX - centroid_y_px) * NM_PER_PIXEL

    return layout_x_nm, layout_y_nm


def bbox_pixel_to_layout_coordinates(tile_x, tile_y, x_min_px, x_max_px, y_min_px, y_max_px):
    """
    Convert predicted bounding box from pixel coordinates to approximate layout coordinates.
    """
    bbox_x_min_nm = tile_x + x_min_px * NM_PER_PIXEL
    bbox_x_max_nm = tile_x + (x_max_px + 1) * NM_PER_PIXEL

    # Because of vertical flip:
    bbox_y_max_nm = tile_y + (IMAGE_SIZE_PX - y_min_px) * NM_PER_PIXEL
    bbox_y_min_nm = tile_y + (IMAGE_SIZE_PX - (y_max_px + 1)) * NM_PER_PIXEL

    return bbox_x_min_nm, bbox_y_min_nm, bbox_x_max_nm, bbox_y_max_nm

### Extract Predicted Violation Coordinates on the Test Set

Here we run the trained U-Net on the held-out test set and convert each predicted DRC mask into numerical layout coordinates.

We only report samples where:
- the ground-truth mask contains a real DRC violation
- the model also predicted a non-empty mask

The output table gives the predicted violation center and bounding box in both pixels and nanometers.

In [ ]:
model.eval()

location_results = []

# test_dataset is a torch Subset, so it stores original indices into seg_dataset
test_indices = test_dataset.indices

with torch.no_grad():
    for original_idx in test_indices:
        # Load sample directly from the original segmentation dataset
        image, gt_mask = seg_dataset[original_idx]

        image_file = seg_dataset.image_files[original_idx]
        tile_x, tile_y = parse_tile_origin_from_filename(image_file)

        # Skip if filename parsing failed
        if tile_x is None:
            continue

        # Run model on one sample
        input_image = image.unsqueeze(0).to(device)  # [1, 1, H, W]

        logits = model(input_image)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob > best_threshold).astype(np.uint8)

        gt_mask_np = gt_mask[0].numpy()

        # We focus on actual DRC violations in the test set
        if gt_mask_np.sum() == 0:
            continue

        pred_location = extract_mask_location(pred_mask, prob_map=prob)
        gt_location = extract_mask_location(gt_mask_np)

        # If model missed the violation entirely
        if pred_location is None:
            location_results.append({
                "tile_file": image_file,
                "tile_x_nm": tile_x,
                "tile_y_nm": tile_y,
                "predicted": False
            })
            continue

        # Predicted centroid in layout coordinates
        pred_layout_x_nm, pred_layout_y_nm = pixel_to_layout_coordinates(
            tile_x,
            tile_y,
            pred_location["centroid_x_px"],
            pred_location["centroid_y_px"]
        )

        # Predicted bbox in layout coordinates
        bbox_x_min_nm, bbox_y_min_nm, bbox_x_max_nm, bbox_y_max_nm = bbox_pixel_to_layout_coordinates(
            tile_x,
            tile_y,
            pred_location["x_min_px"],
            pred_location["x_max_px"],
            pred_location["y_min_px"],
            pred_location["y_max_px"]
        )

        # Ground-truth centroid in layout coordinates, for comparison
        gt_layout_x_nm, gt_layout_y_nm = pixel_to_layout_coordinates(
            tile_x,
            tile_y,
            gt_location["centroid_x_px"],
            gt_location["centroid_y_px"]
        )

        centroid_error_px = np.sqrt(
            (pred_location["centroid_x_px"] - gt_location["centroid_x_px"])**2 +
            (pred_location["centroid_y_px"] - gt_location["centroid_y_px"])**2
        )

        centroid_error_nm = centroid_error_px * NM_PER_PIXEL

        location_results.append({
            "tile_file": image_file,
            "tile_x_nm": tile_x,
            "tile_y_nm": tile_y,
            "predicted": True,

            "pred_centroid_x_px": pred_location["centroid_x_px"],
            "pred_centroid_y_px": pred_location["centroid_y_px"],
            "pred_centroid_x_nm": pred_layout_x_nm,
            "pred_centroid_y_nm": pred_layout_y_nm,

            "bbox_x_min_nm": bbox_x_min_nm,
            "bbox_y_min_nm": bbox_y_min_nm,
            "bbox_x_max_nm": bbox_x_max_nm,
            "bbox_y_max_nm": bbox_y_max_nm,

            "gt_centroid_x_nm": gt_layout_x_nm,
            "gt_centroid_y_nm": gt_layout_y_nm,

            "centroid_error_px": centroid_error_px,
            "centroid_error_nm": centroid_error_nm,

            "confidence": pred_location["confidence"],
            "max_probability": pred_location["max_probability"]
        })

locations_df = pd.DataFrame(location_results)

print("Number of extracted predicted violation locations:", len(locations_df))
locations_df.head(10)

### Example Numerical DRC Prediction

This cell prints one example prediction in a human-readable format.  
The result represents the predicted DRC violation location in approximate layout coordinates.

In [ ]:
example = locations_df[locations_df["predicted"] == True].iloc[0]

print("Example Predicted DRC Violation")
print("--------------------------------")
print(f"Tile file: {example['tile_file']}")
print(f"Tile origin: ({example['tile_x_nm']:.0f} nm, {example['tile_y_nm']:.0f} nm)")
print()
print(f"Predicted center:")
print(f"  x = {example['pred_centroid_x_nm']:.2f} nm")
print(f"  y = {example['pred_centroid_y_nm']:.2f} nm")
print()
print(f"Predicted bounding box:")
print(f"  x_min = {example['bbox_x_min_nm']:.2f} nm")
print(f"  y_min = {example['bbox_y_min_nm']:.2f} nm")
print(f"  x_max = {example['bbox_x_max_nm']:.2f} nm")
print(f"  y_max = {example['bbox_y_max_nm']:.2f} nm")
print()
print(f"Confidence: {example['confidence']:.4f}")
print(f"Max probability: {example['max_probability']:.4f}")
print(f"Centroid error: {example['centroid_error_px']:.2f} px = {example['centroid_error_nm']:.2f} nm")

### Numerical Localization Results

The predicted segmentation masks were converted into numerical DRC violation locations.  
For each non-empty predicted mask, we extracted the centroid, bounding box, and confidence score.  
The centroid and bounding box were converted from pixel coordinates to approximate layout coordinates using the rasterization resolution of 8 nm/pixel.

This step transforms the U-Net output from a visual mask into a practical localization result that can be interpreted as a predicted DRC violation location in the layout.